In [ ]:
import keras
from keras import layers
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
data = np.load("Preprocessed_Data/OTIDS_clean_data.npz")
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]

In [ ]:
#1d cnn sequential model: 
def create_model_otids(): #this is the same model we'll always use for all. 
    model = keras.Sequential()
    model.add(layers.Input(shape=(8, 11)))
    model.add(layers.Conv1D(16, 4, activation='relu'))
    model.add(layers.GlobalMaxPooling1D())
    model.add(layers.Dense(1, activation='sigmoid')) #output 1 bc we only have 2 labels: attack or not attack
    return model
model = create_model_otids()

In [ ]:
b_size = 32
callbacks = [
    ModelCheckpoint("saved_models/best_model_road_cnn.keras", monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-9, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=16, verbose=1, restore_best_weights=True)
]
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss = 'binary_crossentropy', 
              metrics = ['accuracy', keras.metrics.AUC(name='auc'), 
              keras.metrics.Precision(name='precision'),  keras.metrics.Recall(name='recall')])
model.fit(X_train, y_train, batch_size = b_size, epochs = 30, validation_split=0.1, callbacks = callbacks, verbose = 1)

In [ ]:
testing_acc = model.evaluate(X_test,y_test, verbose=1)
print(f"Test loss: {testing_acc[0]}")
print(f"Test accuracy: {testing_acc[1]}")

In [ ]:
#examine classification predictions
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)       
y_true = np.argmax(y_test_cat, axis=1) 
print(classification_report(y_true, y_pred, target_names=["Normal", "DoS", "Fuzzy", "Impersonation"]))

#ROC AUC 
#sensitivity (recall) and 
# validity (precision)
from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(y_test_cat, y_pred_probs, multi_class='ovr')
print(f"ROC AUC Score: {roc_auc}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Normal", "DoS", "Fuzzy", "Impersonation"],
            yticklabels=["Normal", "DoS", "Fuzzy", "Impersonation"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()  